**Import Libraries**

In [5]:
import cv2
import mediapipe as mp
import numpy as np
import joblib
import json
from collections import deque
import time

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

print("Libraries imported")


Libraries imported


**Load Trained Model and scaler**

In [12]:
model = joblib.load('../models/gesture_classifier.pkl')
scaler = joblib.load('../models/scaler.pkl')

# Load metadata
with open('../models/model_metadata.json', 'r') as f:
    metadata = json.load(f)

GESTURE_CLASSES = {int(k): v for k, v in metadata['gesture_classes'].items()}

print(f" Model loaded successfully")
print(f"  Model: {metadata['model_name']}")
print(f"  Accuracy: {metadata['accuracy']:.4f}")
print(f"  Gestures: {list(GESTURE_CLASSES.values())}")

 Model loaded successfully
  Model: Logistic Regression
  Accuracy: 1.0000
  Gestures: ['fist', 'open_palm', 'thumbs_up', 'peace', 'pointing', 'ok_sign']


**Feature Extractor**

In [21]:
class HandFeatureExtractor:
    
    def __init__(self):
        self.finger_tips = [4, 8, 12, 16, 20]
        self.finger_pips = [2, 6, 10, 14, 18]
        self.finger_mcps = [1, 5, 9, 13, 17]
        
    def extract_features(self, hand_landmarks) -> np.ndarray:
        landmarks = self._get_landmark_array(hand_landmarks)
        normalized = self._normalize_landmarks(landmarks)
        distances = self._calculate_distances(landmarks)
        angles = self._calculate_angles(landmarks)
        finger_states = self._get_finger_states(landmarks)
        
        features = np.concatenate([
            normalized.flatten(),
            distances,
            angles,
            finger_states
        ])
        return features
    
    def _get_landmark_array(self, hand_landmarks) -> np.ndarray:
        return np.array([[lm.x, lm.y] for lm in hand_landmarks.landmark])
    
    def _normalize_landmarks(self, landmarks: np.ndarray) -> np.ndarray:
        wrist = landmarks[0]
        normalized = landmarks - wrist
        hand_size = np.max(np.linalg.norm(normalized, axis=1))
        if hand_size > 0:
            normalized = normalized / hand_size
        return normalized
    
    def _calculate_distances(self, landmarks: np.ndarray) -> np.ndarray:
        wrist = landmarks[0]
        return np.array([np.linalg.norm(landmarks[tip] - wrist) 
                        for tip in self.finger_tips])
    
    def _calculate_angles(self, landmarks: np.ndarray) -> np.ndarray:
        angles = []
        for tip, pip, mcp in zip(self.finger_tips, self.finger_pips, self.finger_mcps):
            v1 = landmarks[pip] - landmarks[mcp]
            v2 = landmarks[tip] - landmarks[pip]
            cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-6)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))
            angles.append(angle)
        return np.array(angles)
    
    def _get_finger_states(self, landmarks: np.ndarray) -> np.ndarray:
        wrist = landmarks[0]
        states = []
        for tip, pip in zip(self.finger_tips, self.finger_pips):
            tip_dist = np.linalg.norm(landmarks[tip] - wrist)
            pip_dist = np.linalg.norm(landmarks[pip] - wrist)
            states.append(1 if tip_dist > pip_dist * 1.1 else 0)
        return np.array(states)

feature_extractor = HandFeatureExtractor()

Prediction Smoother (Reduce Jitter)

**Uses majority voting**

In [25]:
class PredictionSmoother:

    def __init__(self, window_size: int = 5):
        self.window_size = window_size
        self.prediction_queue = deque(maxlen=window_size)
    
    def add_prediction(self, prediction: int) -> int:
    
        self.prediction_queue.append(prediction)
        
        # Return common prediction
        if len(self.prediction_queue) > 0:
            return max(set(self.prediction_queue), 
                      key=self.prediction_queue.count)
        return prediction
    
    def reset(self):
        self.prediction_queue.clear()

**Real-Time Gesture Recognition System**

In [30]:
class GestureRecognitionSystem:
    
    def __init__(self, model, scaler, gesture_classes, smooth_window=5):
        self.model = model
        self.scaler = scaler
        self.gesture_classes = gesture_classes
        self.feature_extractor = HandFeatureExtractor()
        self.smoother = PredictionSmoother(window_size=smooth_window)
        
        # Performance tracking
        self.fps_queue = deque(maxlen=30)

In [32]:
   def predict_gesture(self, hand_landmarks):
        # Extract features
        features = self.feature_extractor.extract_features(hand_landmarks)
        
        # Scale features
        features_scaled = self.scaler.transform(features.reshape(1, -1))
        
        # Predict
        prediction = self.model.predict(features_scaled)[0]
        
        # Smooth prediction
        smoothed_prediction = self.smoother.add_prediction(prediction)
        
        # Get confidence if available
        confidence = None
        if hasattr(self.model, 'predict_proba'):
            probabilities = self.model.predict_proba(features_scaled)[0]
            confidence = probabilities[smoothed_prediction]
        
        return smoothed_prediction, confidence

In [34]:
 def draw_ui(self, frame, gesture_name, confidence, fps):
        h, w, _ = frame.shape
        
        # Semi-transparent background
        overlay = frame.copy()
        cv2.rectangle(overlay, (10, 10), (w-10, 150), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
        
        # Gesture name (large)
        cv2.putText(frame, f"Gesture: {gesture_name.upper()}", (20, 50),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
        
        # Confidence bar
        if confidence is not None:
            cv2.putText(frame, f"Confidence: {confidence:.2%}", (20, 90),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            
            # Draw confidence bar
            bar_width = int(300 * confidence)
            cv2.rectangle(frame, (20, 100), (320, 120), (50, 50, 50), -1)
            cv2.rectangle(frame, (20, 100), (20 + bar_width, 120), 
                         (0, 255, 0), -1)
        
        # FPS
        cv2.putText(frame, f"FPS: {fps:.1f}", (20, 140),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Instructions
        cv2.putText(frame, "Press 'q' to quit", (w-200, h-20),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

In [61]:
  def run(self):
        cap = cv2.VideoCapture(0)
        hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7,
            min_tracking_confidence=0.7
        )
        
        print(" REAL-TIME GESTURE RECOGNITION")
        print("\nShow your hand and make gestures!")
        print("Press 'q' to quit\n")
        
        prev_time = time.time()
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            # Calculate FPS
            curr_time = time.time()
            fps = 1 / (curr_time - prev_time + 1e-6)
            prev_time = curr_time
            self.fps_queue.append(fps)
            avg_fps = sum(self.fps_queue) / len(self.fps_queue)
            
            # Flip for mirror view
            frame = cv2.flip(frame, 1)
            
            # Convert to RGB
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Process frame
            results = hands.process(rgb_frame)

            if results.multi_hand_landmarks:
                hand_landmarks = results.multi_hand_landmarks[0]
        
                
                # Draw hand landmarks
                mp_drawing.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style()
                )
                
                # Predict gesture
                prediction, confidence = self.predict_gesture(hand_landmarks)
                gesture_name = self.gesture_classes[prediction]
                
                # Draw UI
                self.draw_ui(frame, gesture_name, confidence, avg_fps)
            else:
                # No hand detected
                cv2.putText(frame, "No hand detected", (20, 50),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                self.smoother.reset()
            
            # Display
            cv2.imshow('Gesture Recognition', frame)
            
            # Quit on 'q'
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()
        hands.close()
        
        print("\n Recognition stopped")
        print(f"Average FPS: {avg_fps:.1f}")

In [64]:
class GestureRecognitionSystem:
    def __init__(self, model, scaler, gesture_classes, smooth_window=5):
        self.model = model
        self.scaler = scaler
        self.gesture_classes = gesture_classes
        self.feature_extractor = HandFeatureExtractor()
        self.prediction_buffer = []

        self.smooth_window = smooth_window

        self.hands = mp.solutions.hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=0.7,
            min_tracking_confidence=0.7
        )

    def run(self):
        cap = cv2.VideoCapture(0)

        print(" Real-Time Gesture Recognition")
        print("Press 'q' to quit")

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = self.hands.process(rgb)

            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    mp.solutions.drawing_utils.draw_landmarks(
                        frame,
                        hand_landmarks,
                        mp.solutions.hands.HAND_CONNECTIONS
                    )

                    features = self.feature_extractor.extract_features(hand_landmarks)
                    features = features.reshape(1, -1)
                    features = self.scaler.transform(features)

                    pred = self.model.predict(features)[0]

                    self.prediction_buffer.append(pred)
                    if len(self.prediction_buffer) > self.smooth_window:
                        self.prediction_buffer.pop(0)

                    final_pred = max(
                        set(self.prediction_buffer),
                        key=self.prediction_buffer.count
                    )

                    gesture_name = self.gesture_classes[final_pred]

                    cv2.putText(
                        frame,
                        f"Gesture: {gesture_name}",
                        (30, 50),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1.2,
                        (0, 255, 0),
                        3
                    )

            cv2.imshow("Gesture Recognition", frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.hands.close()

**Run Real-Time Recognition**

In [67]:
recognition_system = GestureRecognitionSystem(
    model=model,
    scaler=scaler,
    gesture_classes=GESTURE_CLASSES,
    smooth_window=5
)

# Run recognition
recognition_system.run()

I0000 00:00:1766775719.825454       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2


 Real-Time Gesture Recognition
Press 'q' to quit
